In [1]:
import json
import utils
import pandas as pd
from dotenv import load_dotenv
_=load_dotenv()


In [3]:
!pip install aisuite

  Using cached aisuite-0.2.0-py3-none-any.whl.metadata (12 kB)
Using cached aisuite-0.2.0-py3-none-any.whl (228 kB)



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import aisuite as ai
client = ai.Client()

In [5]:
utils.create_transactions_db()

SQLite database 'products.db' created with a single 'transactions' table (event-sourced).


In [6]:
utils.print_html(utils.get_schema('products.db'))

In [10]:
def generate_sql_query(question: str, schema: str, model: str)->str:
    prompt = f"""
    You are a sql assistant. Given the schema and the users question, write a sql query for sqllite.
    
    Schema:
    {schema}
    
    User Question:
    {question}

    Respond with SQL only.
    """
    response = client.chat.completions.create(
        model = model,
        messages = [{"role":"user", "content":prompt}],
        temperature = 0,
    )
    return response.choices[0].message.content.strip()

In [11]:
# Example usage of generate_sql

# We provide the schema as a string
schema = """
Table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""

# We ask a question about the data in natural language
question = "Which color of product has the highest total sales?"

utils.print_html(question, title="User Question")

# Generate the SQL query using the specified model
sql_V1 = generate_sql_query(question, schema, model="openai:gpt-4.1")

# Display the generated SQL query
utils.print_html(sql_V1, title="SQL Query V1")

In [12]:
df_sql_v1 = utils.execute_sql(sql_V1, db_path='products.db')
utils.print_html(df_sql_v1, title = "Output of SQL query V1 - ❌ Does NOT fully answer the question")

color,total_sales
blue,-190571.46


In [13]:
def refine_sql(
        question: str,
        sql_query: str,
        schema: str,
        model: str,
)-> tuple[str,str]:
    """Relect on whether a query's *showm output* answers the question,
    and propose an improved SQl if neeeded.
    Return(feedback, refined_sql)"""
    prompt = f"""
    You are a SQL reviewer and refiner.

    User asked:
    {question}
    Schema:
    {schema}
    Original_SQL:
    {sql_query}

    Step1: Breifely evealuate if the SQL OUTPUT fully answers the user question.
    Step2: If improvement needed, please provide a refined sql query for SQLlite.
    If the original SQL is already correct return it unchanged.

    Return STRICT JSON with two fields:
    {{
        "feedback": "<1-3 Sentences explaining the gap or confirming correctness>",
        "refined_sql": "<final SQl to run>"
    }}

    """
    response = client.chat.completions.create(
        model = model,
        messages = [{"role": "user", "content": prompt}],
        temperature = 0
    )

    content = response.choices[0].message.content
    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback","")).strip()
        refined_sql = str(obj.get("refined_sql",sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        feedback = content.strip()
        refined_sql = sql_query
    return feedback, refined_sql

In [15]:
feedback, sql_V2 = refine_sql(
    question= question,
    sql_query= sql_V1,
    schema=schema,
    model="openai:gpt-4.1"
)
utils.print_html(question, title = "User Question")
utils.print_html(sql_V1, title = "generated SQL Query (V1)")
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')
utils.print_html(df_sql_v1,title= "utput of V1 - ❌ Does NOT fully answer the question")

utils.print_html(feedback, title = "Feedback on V1")
utils.print_html(sql_V2,title="Refined SQL query V2")
df_sql_V2 =utils.execute_sql(sql_V2, db_path= 'products.db')
utils.print_html(df_sql_V2, title="SQL Output of V2 - ❌ Does NOT fully answer the question")


color,total_sales
blue,-190571.46


color,total_sales
blue,-190571.46


In [ ]:
def refine_sql_external_feedback(
    question: str,
    sql_query: str,
    df_feedback: pd.DataFrame,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """
    Evaluate whether the SQL result answers the user's question and,
    if necessary, propose a refined version of the query.
    Returns (feedback, refined_sql).
    """
    prompt = f"""
    You are a SQL reviewer and refiner.

    User asked:
    {question}

    Original SQL:
    {sql_query}

    SQL Output:
    {df_feedback.to_markdown(index=False)}

    Table Schema:
    {schema}
    Step 1: Before judging the SQL, check its OUTPUT:
    - Are any values negative, zero, NULL or implausible for the question?
    (for example, sales totals should not be negative)
    - Does the sort direction return what the question asks for?
    Step 2: If any check fails, fix the query. Return it unchanged only if all checks pass.
    Step 3: Briefly evaluate if the SQL output answers the user's question.
    Step 4: If the SQL could be improved, provide a refined SQL query.
    If the original SQL is already correct, return it unchanged.

    Return a strict JSON object with two fields:
    - "feedback": brief evaluation and suggestions
    - "refined_sql": the final SQL to run
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    
    content = response.choices[0].message.content
    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        # Fallback if the model does not return valid JSON:
        # use the raw content as feedback and keep the original SQL
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql

In [21]:
# Example: Refine SQL with External Feedback (V1 → V2)

# Execute the original SQL (V1)
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')

# Use external feedback to evaluate and refine
feedback, sql_V2 = refine_sql_external_feedback(
    question=question,
    sql_query=sql_V1,   # V1 query
    df_feedback=df_sql_V1,    # Output of V1
    schema=schema,
    model="openai:gpt-4.1"
)

# --- V1 ---
utils.print_html(question, title="User Question")
utils.print_html(sql_V1, title="Generated SQL Query (V1)")
utils.print_html(df_sql_V1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

# --- Feedback & V2 ---
utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_V2, title="Refined SQL Query (V2)")

# Execute and display V2 results
df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="SQL Output of V2 (with External Feedback) - ✅ Fully answers the question")


color,total_sales
blue,-190571.46


color,total_sales
white,358315.09


In [28]:
def run_sql_workflow(
        db_path: str,
        question: str,
        model_generation: str = "openai:gpt-4.1",
        model_evaluation: str = "openai:gpt-4.1",
):
    schema = utils.get_schema(db_path)
    utils.print_html(schema,title="Extract Database Schema")

    sql_v1 = generate_sql_query(question,schema,model_generation)
    utils.print_html(sql_v1, title="Generated SQL (V1)")

    df_v1 = utils.execute_sql(sql_v1,db_path)
    utils.print_html(df_v1, title="SQL output (V1)")

    feedback, sql_v2 = refine_sql_external_feedback(question, sql_v1,df_v1,schema, model_evaluation)

    utils.print_html(feedback, title="Feedback for SQL Query V1")
    utils.print_html(sql_v2,title="Refined SQl(V2)")

    df_v2 = utils.execute_sql(sql_v2,db_path)
    utils.print_html(df_v2, title="Generated Output from SQL Query V2")

In [50]:
run_sql_workflow(
    "products.db","Which Color of product has the highest total Sales?",
    model_generation="openai:gpt-4.1",
    model_evaluation="openai:gpt-4.1"
)

color,total_sales
blue,-190571.46


color,total_sales
white,358315.09


In [51]:
f=utils.execute_sql("SELECT action, COUNT(*) AS n, MIN(qty_delta) AS min_qty, MAX(qty_delta) AS max_qty FROM transactions GROUP BY action", "products.db")
utils.print_html(f, title="total Count")

action,n,min_qty,max_qty
insert,100,5,50
price_update,723,0,0
restock,1258,1,25
sale,2919,-10,-1
